In [ ]:
# 1. Khởi tạo môi trường và Module
import Pkg
Pkg.activate("..")

if !isdefined(Main, :Algorithm)
    include("../src/algorithm/relim.jl")
    using .Algorithm
    using .Utils
end

using Printf

# Danh sách các tập tin Benchmark
datasets = [
    "../data/benchmark/retail.txt",
    "../data/benchmark/mushrooms.txt",
    "../data/benchmark/T10I4D100K.txt",
    "../data/benchmark/accidents.txt"
]

# TỪ ĐIỂN: Cấu hình Minsup cực chuẩn đánh dấu cho từng File
baseline_minsup = Dict(
    "retail.txt" => 0.05,          
    "mushrooms.txt" => 0.30,       
    "T10I4D100K.txt" => 0.05,      
    "accidents.txt" => 0.60        
)

println("="^50)
println(" BẮT ĐẦU BENCHMARK VÀ XUẤT OUTPUT ")
println("="^50)

for filepath in datasets
    dataset_name = basename(filepath)
    println("\n[▶] Đang xử lý: ", dataset_name)
    
    # Ép Julia dọn rác trước khi chạy tập mới
    GC.gc() 
    
    # -- 1. ĐỌC DỮ LIỆU --
    local transactions
    try
        transactions = read_spmf_file(filepath)
    catch e
        println("   -> [CẢNH BÁO]: Không tìm thấy file '", dataset_name, "'. Trượt qua tập khác!")
        continue
    end
    
    n_trans = length(transactions)
    minsup_val = get(baseline_minsup, dataset_name, 0.50)
    minsup_count = Int(ceil(minsup_val * n_trans))
    
    println("   - Base Minsup     : ", minsup_val * 100, "%")
    println("   - Số giao dịch    : ", n_trans)
    
    # -- 2. CHẠY RELIM VÀ ĐO ĐẠC BỘ NHỚ --
    stats = @timed begin
        itemsets = relim_mine(transactions, minsup_count)
    end
    
    # -- 3. TÍNH TOÁN VÀ IN KẾT QUẢ --
    time_sec = stats.time
    alloc_mb = stats.bytes / (1024^2) 
    peak_rss_mb = Sys.maxrss() / (1024^2)
    
    @printf("   - Thời gian chạy  : %.4f giây\n", time_sec)
    @printf("   - Mức RAM Tối đa  : %.2f MB (Peak RAM)\n", peak_rss_mb)
    println("   - Tổng (Itemsets) : ", length(itemsets))

    # -- 4. THAO TÁC GHI FILE KẾT QUẢ VÀO FOLDER NOTEBOOK --
    # Tự động xuất ra file cùng cấp thư mục hiện tại của bạn
    output_filename = "output_" * dataset_name
    println("   - Đang ghi file   : ", output_filename, " ...")
    
    # Sắp xếp các itemset ngắn lên đầu cho dễ nhìn trước khi ghi
    write_spmf_file(output_filename, itemsets)
    println("   - Xuất file       : THÀNH CÔNG!")
    
    # -- 5. CHỐNG TRÀN RAM --
    transactions = nothing
    itemsets = nothing
    stats = nothing
    GC.gc() 
end

println("\n", "="^50)
println(" HOÀN TẤT BENCHMARK! Vui lòng kiểm tra mục lục folder.")


  Activating project at `d:\Nam 3\Ki 2\Data Mining\LAB\Lab 02\DM_Lab02\11`



🔥 BẮT ĐẦU BENCHMARK VÀ XUẤT OUTPUT 🔥

[▶] Đang xử lý: retail.txt
   - Base Minsup     : 5.0%
   - Số giao dịch    : 88162
   - Thời gian chạy  : 0.0277 giây
   - Mức RAM Tối đa  : 401.80 MB (Peak RAM)
   - Tổng (Itemsets) : 16
   - Đang ghi file   : output_retail.txt ...
   - Xuất file       : THÀNH CÔNG!

[▶] Đang xử lý: mushrooms.txt
   - Base Minsup     : 30.0%
   - Số giao dịch    : 8416
   - Thời gian chạy  : 0.1348 giây
   - Mức RAM Tối đa  : 444.12 MB (Peak RAM)
   - Tổng (Itemsets) : 2587
   - Đang ghi file   : output_mushrooms.txt ...
   - Xuất file       : THÀNH CÔNG!

[▶] Đang xử lý: T10I4D100K.txt
   - Base Minsup     : 5.0%
   - Số giao dịch    : 100000
   - Thời gian chạy  : 0.0290 giây
   - Mức RAM Tối đa  : 444.12 MB (Peak RAM)
   - Tổng (Itemsets) : 10
   - Đang ghi file   : output_T10I4D100K.txt ...
   - Xuất file       : THÀNH CÔNG!

[▶] Đang xử lý: accidents.txt
   - Base Minsup     : 60.0%
   - Số giao dịch    : 340183
   - Thời gian chạy  : 24.1823 giây
   - Mức 